# Data preprocessing — CIC-IoT2023 (paper Sec. 4.2)

Two main steps:
1. Bidirectional MAC filter (paper Sec. 4.2 step 1).
2. Combine per-attack-type CSVs into a class-balanced train/test split
   targeting 20,000 train / 4,000 test per class via under/oversampling
   (paper Sec. 4.2 step 2; numbers match Table 2).

Pre-requisite: NFStream + multi-scale temporal extraction must already
have produced one CSV per attack type (`<class>-<...>.csv`).

In [ ]:
import glob, os
from Utility.Functions import (
    split_csv, Combining_classes, DEFAULT_CIC_IOT2023_LABELS,
    duplicate_rows, random_pick_rows, standardize_flow_features,
)
import pandas as pd

CSV_DIR = r'F:/CIC_IOT/Extracted_Flow_Features/'   # <-- edit
OUTPUT_TRAIN = os.path.join(CSV_DIR, 'train', 'df_class_8_train.csv')
OUTPUT_TEST  = os.path.join(CSV_DIR, 'train', 'df_class_8_test.csv')

## Step 1 — Bidirectional MAC filter

Paper Sec. 4.2: a flow is kept only if its source or destination MAC
matches an attacker MAC (for attack rows), and conversely benign flows
are dropped if they touch any attacker MAC. The 9-MAC list is from
paper Sec. 4.2 (also reproduced in [`Utility/Functions.py`](Utility/Functions.py)
as `CIC_IOT2023_ATTACKER_MACS`).

In [ ]:
for csv in glob.glob(os.path.join(CSV_DIR, '*.csv')):
    if 'train' in csv or 'test' in csv:
        continue
    print('split_csv', os.path.basename(csv))
    split_csv(csv, test_sample=4000, number_in_individual_class=20000)

## Step 2 — Per-class balance + assemble final CSVs

Targets paper Table 2 numbers (training data column):
  Benign       1,045,581 → undersample to 20,000
  DDoS         26,510,228 → undersample to 20,000
  ...
  WebBased     20,000 (oversampled from 5,449)
  BruteForce   20,000 (oversampled from 2,336)

In [ ]:
Combining_classes(
    directory=CSV_DIR,
    classes_list=list(DEFAULT_CIC_IOT2023_LABELS.keys()),
    Number_in_individaul_class=20000,
    Number_of_test_samples=4000,
)

## Step 3 — Stitch per-class train/test CSVs into a single file

`Combining_classes` writes one `<class>_train.csv` and one `<class>_test.csv`
per class. Concatenate them into the unified files consumed by
`NIDSDataset`.

In [ ]:
train_dir = os.path.join(CSV_DIR, 'train')
train_dfs = [pd.read_csv(p) for p in glob.glob(os.path.join(train_dir, '*_train.csv'))]
test_dfs  = [pd.read_csv(p) for p in glob.glob(os.path.join(train_dir, '*_test.csv'))]
train_df = pd.concat(train_dfs, ignore_index=True).sample(frac=1.0, random_state=42)
test_df  = pd.concat(test_dfs,  ignore_index=True).sample(frac=1.0, random_state=42)
print('train rows:', len(train_df), 'test rows:', len(test_df))
print('train class counts:\n', train_df['Label'].value_counts())
print('test class counts:\n',  test_df['Label'].value_counts())

## Step 4 — Standardize numeric columns (paper eq. 14)

Apply `(x - mu) / sigma` using statistics from the train split, then
apply the same shift/scale to the test split.

In [ ]:
ignore = {'Label'}
ignore |= {c for c in train_df.columns if c.startswith('udps.')}
train_df, test_df = standardize_flow_features(train_df, test_df, ignore_cols=ignore)

train_df.to_csv(OUTPUT_TRAIN, index=False)
test_df.to_csv(OUTPUT_TEST,  index=False)
print('Wrote', OUTPUT_TRAIN, OUTPUT_TEST)